# Resultats de les automatitzacions sobre 180 sèries

Aquest notebook resumeix els resultats obtinguts amb les dues automatitzacions del TFG: els models SARIMA i els models híbrids SARIMA-NNAR. 

L'objectiu és obtenir taules per complementar l'anàlisi detallada de les sèries representatives.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

SARIMA_PATH = Path("results_sarima.csv")
HYBRID_PATH = Path("results_hybrid.csv")

sarima = pd.read_csv(SARIMA_PATH)
hybrid = pd.read_csv(HYBRID_PATH)

resum_carrega = pd.DataFrame({
    "Fitxer": [SARIMA_PATH.name, HYBRID_PATH.name],
    "Nombre de files": [len(sarima), len(hybrid)],
    "Nombre de columnes": [sarima.shape[1], hybrid.shape[1]]
})

resum_carrega

,Fitxer,Nombre de files,Nombre de columnes
0,results_sarima.csv,180,19
1,results_hybrid.csv,174,29


## Preparació de les sèries

S'exclou el diagnòstic COVID i es conserven només els ajustos amb `status == "ok"`.

In [2]:
KEYS = ["age", "diag", "region"]

sarima_no_covid = sarima.loc[~sarima["diag"].str.contains("COVID", case=False, na=False)].copy()
hybrid_no_covid = hybrid.loc[~hybrid["diag"].str.contains("COVID", case=False, na=False)].copy()

sarima_ok = sarima_no_covid.loc[sarima_no_covid["status"] == "ok"].copy()
hybrid_ok = hybrid_no_covid.loc[hybrid_no_covid["status"] == "ok"].copy()

resum_preparacio = pd.DataFrame({
    "Indicador": [
        "Sèries SARIMA sense COVID",
        "Sèries SARIMA correctes",
        "Sèries híbrides sense COVID",
        "Sèries híbrides correctes",
    ],
    "Valor": [
        len(sarima_no_covid),
        len(sarima_ok),
        len(hybrid_no_covid),
        len(hybrid_ok),
    ],
})

resum_preparacio

,Indicador,Valor
0,Sèries SARIMA sense COVID,160
1,Sèries SARIMA correctes,160
2,Sèries híbrides sense COVID,160
3,Sèries híbrides correctes,160


## Unió dels resultats SARIMA i SARIMA-NNAR

Les dues taules s'uneixen per grup d'edat, diagnòstic i regió.

In [3]:
sarima_comp = sarima_ok[
    KEYS + ["mae_test", "rmse_test", "cov_total", "cov_covid", "cov_post"]
].rename(columns={
    "mae_test": "sarima_mae",
    "rmse_test": "sarima_rmse",
    "cov_total": "sarima_cov_total",
    "cov_covid": "sarima_cov_covid",
    "cov_post": "sarima_cov_post",
})

hybrid_comp = hybrid_ok[
    KEYS + [
        "hybrid_mae",
        "hybrid_rmse",
        "hybrid_cov_total",
        "hybrid_cov_covid",
        "hybrid_cov_post",
    ]
].copy()

df = hybrid_comp.merge(sarima_comp, on=KEYS, how="inner")

resum_unio = pd.DataFrame({
    "Indicador": ["Sèries comparables SARIMA vs SARIMA-NNAR"],
    "Valor": [len(df)],
})

resum_unio

,Indicador,Valor
0,Sèries comparables SARIMA vs SARIMA-NNAR,160


## Càlcul de millores del model híbrid

La millora es calcula com la diferència entre l'error del SARIMA i l'error del model híbrid. 

In [4]:
for metric in ["mae", "rmse"]:
    df[f"delta_{metric}"] = df[f"sarima_{metric}"] - df[f"hybrid_{metric}"]
    df[f"delta_{metric}_pct"] = 100 * df[f"delta_{metric}"] / df[f"sarima_{metric}"]
    df[f"millora_{metric}"] = df[f"delta_{metric}"] > 0

df["covid_menor_que_post_sarima"] = df["sarima_cov_covid"] < df["sarima_cov_post"]
df["covid_menor_que_post_hybrid"] = df["hybrid_cov_covid"] < df["hybrid_cov_post"]

## Taula final per al TFG

In [5]:
taula_tfg = pd.DataFrame({
    "Resultat": [
        "Sèries SARIMA correctes",
        "Sèries híbrides correctes",
        "Sèries comparables",
        "Millora MAE del model híbrid",
        "Millora RMSE del model híbrid",
        "Millora mediana MAE",
        "Millora mediana RMSE",
        "Cobertura mediana COVID SARIMA",
        "Cobertura mediana COVID híbrid",
        "Cobertura mediana post-COVID SARIMA",
        "Cobertura mediana post-COVID híbrid",
        "Cobertura COVID inferior a post-COVID, SARIMA",
        "Cobertura COVID inferior a post-COVID, híbrid",
    ],
    "Valor": [
        len(sarima_ok),
        len(hybrid_ok),
        len(df),
        f"{df['millora_mae'].mean() * 100:.1f}%",
        f"{df['millora_rmse'].mean() * 100:.1f}%",
        f"{df['delta_mae_pct'].median():.2f}%",
        f"{df['delta_rmse_pct'].median():.2f}%",
        f"{df['sarima_cov_covid'].median() * 100:.1f}%",
        f"{df['hybrid_cov_covid'].median() * 100:.1f}%",
        f"{df['sarima_cov_post'].median() * 100:.1f}%",
        f"{df['hybrid_cov_post'].median() * 100:.1f}%",
        f"{df['covid_menor_que_post_sarima'].mean() * 100:.1f}%",
        f"{df['covid_menor_que_post_hybrid'].mean() * 100:.1f}%",
    ],
})

taula_tfg


,Resultat,Valor
0,Sèries SARIMA correctes,160
1,Sèries híbrides correctes,160
2,Sèries comparables,160
3,Millora MAE del model híbrid,53.8%
4,Millora RMSE del model híbrid,54.4%
5,Millora mediana MAE,0.12%
6,Millora mediana RMSE,0.25%
7,Cobertura mediana COVID SARIMA,83.7%
8,Cobertura mediana COVID híbrid,88.9%
9,Cobertura mediana post-COVID SARIMA,92.3%


## Taules complementàries

In [6]:
resum_errors = pd.DataFrame({
    "Mètrica": ["MAE", "RMSE"],
    "Mediana SARIMA": [df["sarima_mae"].median(), df["sarima_rmse"].median()],
    "Mediana híbrid": [df["hybrid_mae"].median(), df["hybrid_rmse"].median()],
    "Millora mediana (%)": [df["delta_mae_pct"].median(), df["delta_rmse_pct"].median()],
    "Sèries on millora (%)": [df["millora_mae"].mean() * 100, df["millora_rmse"].mean() * 100],
}).round(2)

resum_cobertura = pd.DataFrame({
    "Període": ["Total", "COVID", "Post-COVID"],
    "SARIMA mediana": [
        df["sarima_cov_total"].median(),
        df["sarima_cov_covid"].median(),
        df["sarima_cov_post"].median(),
    ],
    "Híbrid mediana": [
        df["hybrid_cov_total"].median(),
        df["hybrid_cov_covid"].median(),
        df["hybrid_cov_post"].median(),
    ],
}).round(3)

display(resum_errors)
display(resum_cobertura)

,Mètrica,Mediana SARIMA,Mediana híbrid,Millora mediana (%),Sèries on millora (%)
0,MAE,350.81,353.13,0.12,53.75
1,RMSE,464.28,469.16,0.25,54.37


,Període,SARIMA mediana,Híbrid mediana
0,Total,0.873,0.906
1,COVID,0.837,0.889
2,Post-COVID,0.923,0.942


## Rànquings de sèries

In [7]:
ranking_cols = KEYS + [
    "sarima_rmse",
    "hybrid_rmse",
    "delta_rmse_pct",
    "sarima_cov_covid",
    "hybrid_cov_covid",
]

ranking_millora_rmse = (
    df.sort_values("delta_rmse_pct", ascending=False)
      .loc[:, ranking_cols]
      .head(10)
)

ranking_empitjora_rmse = (
    df.sort_values("delta_rmse_pct", ascending=True)
      .loc[:, ranking_cols]
      .head(10)
)

print("Sèries on més millora el RMSE amb el model híbrid")
display(ranking_millora_rmse)

print("Sèries on més empitjora el RMSE amb el model híbrid")
display(ranking_empitjora_rmse)

Sèries on més millora el RMSE amb el model híbrid


,age,diag,region,sarima_rmse,hybrid_rmse,delta_rmse_pct,sarima_cov_covid,hybrid_cov_covid
154,15+,PneumÃ²nia,Camp de Tarragona,61.394517,57.152466,6.909495,0.942308,0.980769
99,15+,Bronquiolitis,Terres de l'Ebre,89.334436,84.426402,5.494000,0.942308,0.971154
142,15+,Impetigen,Barcelona Metropolitana Nord,70.441089,66.645875,5.387784,0.980769,1.000000
30,0-14,Faringoamigdalitis,Alt Pirineu i Aran,813.241314,770.032595,5.313148,0.576923,0.846154
7,0-14,Altres IRA,Lleida,3453.296169,3281.297296,4.980716,0.586538,0.634615
8,0-14,Altres IRA,PenedÃ¨s,3272.646144,3110.168733,4.964711,0.615385,0.663462
9,0-14,Altres IRA,Terres de l'Ebre,2308.887974,2204.644118,4.514894,0.653846,0.634615
5,0-14,Altres IRA,Catalunya Central,4837.303049,4642.148027,4.034377,0.548077,0.596154
111,15+,Faringoamigdalitis,Barcelona Ciutat,865.358152,830.511523,4.026845,0.250000,0.288462
156,15+,PneumÃ²nia,Girona,64.413540,61.834477,4.003915,0.875000,0.951923


Sèries on més empitjora el RMSE amb el model híbrid


,age,diag,region,sarima_rmse,hybrid_rmse,delta_rmse_pct,sarima_cov_covid,hybrid_cov_covid
95,15+,Bronquiolitis,Catalunya Central,93.963677,110.443374,-17.538369,0.951923,0.990385
70,0-14,PneumÃ²nia,Alt Pirineu i Aran,745.664859,825.518636,-10.709071,0.932692,0.961538
15,0-14,Bronquiolitis,Catalunya Central,1573.100750,1715.550693,-9.055360,0.932692,0.951923
10,0-14,Bronquiolitis,Alt Pirineu i Aran,1814.043825,1949.512511,-7.467774,0.942308,0.951923
124,15+,Faringoamigdalitis estreptocÃ²ccica,Camp de Tarragona,114.414744,122.418625,-6.995498,0.442308,0.625000
29,0-14,Escarlatina,Terres de l'Ebre,339.948254,362.897180,-6.750712,0.961538,0.971154
40,0-14,Faringoamigdalitis estreptocÃ²ccica,Alt Pirineu i Aran,581.118000,619.087891,-6.533938,0.903846,0.913462
92,15+,Bronquiolitis,Barcelona Metropolitana Nord,45.033339,47.686530,-5.891614,0.942308,0.971154
147,15+,Impetigen,Lleida,136.038781,143.806805,-5.710154,0.942308,0.951923
110,15+,Faringoamigdalitis,Alt Pirineu i Aran,141.004791,149.040267,-5.698725,0.701923,0.855769
